# 카페 평균 수명 및 생존율 비교 (프랜차이즈 vs 개인카페)
**팀:** 5조 &nbsp;|&nbsp; **팀원:** 손정안, 최보금, 김성빈, 정지우, 최예정

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
# 한글 폰트 설정
import subprocess, sys
try:
    fm.findfont('NanumGothic', fallback_to_default=False)
except:
    subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], capture_output=True)
    fm._load_fontmanager(try_read_cache=False)

# 사용 가능한 한글 폰트 자동 선택
_korean_fonts = ['NanumGothic', 'NanumBarunGothic', 'Malgun Gothic', 'AppleGothic', 'UnDotum']
_selected_font = None
for _f in _korean_fonts:
    try:
        if fm.findfont(_f, fallback_to_default=False):
            _selected_font = _f
            break
    except:
        pass
if _selected_font is None:
    _selected_font = 'DejaVu Sans'
plt.rcParams['font.family'] = _selected_font
plt.rcParams['axes.unicode_minus'] = False
print(f"사용 폰트: {_selected_font}")

## 1. 강남구 분석

## 1. 라이브러리 & 데이터 로딩

In [ ]:
# ✅ 파일명 수정 (언더바 사용)
data_gn = pd.read_csv('서울시 강남구 휴게음식점 인허가 정보.csv', encoding='EUC-KR')
print("총 데이터 수:", len(data_gn))
data_gn.head()

## 2. 전처리 & 프랜차이즈 분류

In [ ]:
cols = ['사업장명', '상세영업상태명', '인허가일자', '폐업일자', '도로명주소', '좌표정보(X)', '좌표정보(Y)']
data_gn = data_gn[cols].copy()

# 날짜 변환
data_gn['인허가일자'] = pd.to_datetime(data_gn['인허가일자'], errors='coerce')
data_gn['폐업일자']  = pd.to_datetime(data_gn['폐업일자'],  errors='coerce')

# ✅ 프랜차이즈 키워드 수정 (실제 데이터 기준)
franchise_keywords = {
    '스타벅스': '스타벅스',
    '메가커피':  '메가엠지씨',   # ✅ 수정: '메가엠지씨커피' 실제 이름
    '컴포즈':   '컴포즈',
    '빽다방':   '빽다방',
    '투썸':     '투썸플레이스',  # ✅ 수정: '투썸플레이스' 실제 이름
}

def classify(name):
    for brand, keyword in franchise_keywords.items():
        if keyword in str(name):
            return brand
    return '개인카페'

data_gn['브랜드'] = data_gn['사업장명'].apply(classify)

# 브랜드별 수량 확인
print(data_gn['브랜드'].value_counts())

## 3. 브랜드별 추출

In [ ]:
# 스타벅스
starbucks = data_gn[data_gn['브랜드'] == '스타벅스'].copy()
print(f"강남구 스타벅스 총 {len(starbucks)}개")
print(starbucks[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 메가커피
mega = data_gn[data_gn['브랜드'] == '메가커피'].copy()
print(f"강남구 메가커피 총 {len(mega)}개")
print(mega[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 컴포즈
compose = data_gn[data_gn['브랜드'] == '컴포즈'].copy()
print(f"강남구 컴포즈 총 {len(compose)}개")
print(compose[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 빽다방
baek = data_gn[data_gn['브랜드'] == '빽다방'].copy()
print(f"강남구 빽다방 총 {len(baek)}개")
print(baek[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

## 4. 생존율 분석

In [ ]:
brands = ['스타벅스', '메가커피', '컴포즈', '빽다방', '투썸', '개인카페']
result_gn = []

for brand in brands:
    df_b = data_gn[data_gn['브랜드'] == brand]
    total  = len(df_b)
    active = (df_b['상세영업상태명'] == '영업').sum()
    closed = (df_b['상세영업상태명'] == '폐업').sum()
    survival_rate_gn = active / total * 100 if total > 0 else 0

    # 평균 영업 기간 (폐업한 가게 기준)
    closed_df = df_b[df_b['폐업일자'].notna() & df_b['인허가일자'].notna()]
    avg_days = (closed_df['폐업일자'] - closed_df['인허가일자']).dt.days.mean()

    result_gn.append({
        '브랜드': brand,
        '전체': total,
        '영업중': active,
        '폐업': closed,
        '생존율(%)': round(survival_rate_gn, 1),
        '평균영업기간(일)': round(avg_days, 0) if pd.notna(avg_days) else None
    })

df_result_gn = pd.DataFrame(result_gn)
print(df_result_gn.to_string(index=False))

## 5. 시각화 (4종 그래프)

In [ ]:
colors = {
    '스타벅스': '#00704A',
    '메가커피':  '#FF6B35',
    '컴포즈':   '#6A1B9A',
    '빽다방':   '#FFEA00',
    '투썸':     '#FF0000',
    '개인카페':  '#90A4AE'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('강남구 카페 프랜차이즈 vs 개인카페 분석', fontsize=16, fontweight='bold')

# 그래프 1: 브랜드별 생존율
ax = axes[0][0]
bars = ax.bar(df_result_gn['브랜드'], df_result_gn['생존율(%)'],
              color=[colors[b] for b in df_result_gn['브랜드']])
for bar, val in zip(bars, df_result_gn['생존율(%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', fontsize=10)
ax.set_title('브랜드별 생존율 비교')
ax.set_ylabel('생존율 (%)')
ax.set_ylim(0, 115)
ax.tick_params(axis='x', rotation=15)

# 그래프 2: 영업 vs 폐업 누적 막대
ax = axes[0][1]
x = range(len(df_result_gn))
ax.bar(x, df_result_gn['영업중'], label='영업중', color='#2D7A5E')
ax.bar(x, df_result_gn['폐업'], label='폐업', color='#E53935',
       bottom=df_result_gn['영업중'])
ax.set_xticks(x)
ax.set_xticklabels(df_result_gn['브랜드'], rotation=15)
ax.set_title('브랜드별 영업/폐업 현황')
ax.legend()

# 그래프 3: 평균 영업 기간
ax = axes[1][0]
df_days = df_result_gn.dropna(subset=['평균영업기간(일)'])
ax.barh(df_days['브랜드'], df_days['평균영업기간(일)'],
        color=[colors[b] for b in df_days['브랜드']])
ax.set_title('폐업 카페 평균 영업기간 (일)')
ax.set_xlabel('일수')

# 그래프 4: 연도별 신규 개업 추이
ax = axes[1][1]
data_gn['개업연도'] = data_gn['인허가일자'].dt.year
yearly = data_gn.groupby(['개업연도', '브랜드']).size().unstack(fill_value=0)
for brand in brands:
    if brand in yearly.columns:
        ax.plot(yearly.index, yearly[brand], marker='o',
                color=colors[brand], label=brand, linewidth=1.8)
ax.set_title('연도별 신규 개업 추이')
ax.set_xlabel('연도')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('강남구_카페_분석.png', dpi=150)
plt.show()
print("그래프 저장 완료: 강남구_카페_분석.png")

In [ ]:
colors = {
    '스타벅스': '#00704A', '메가커피': '#FF6B35',
    '컴포즈': '#6A1B9A', '빽다방': '#FFEA00',
    '투썸': '#FF0000', '개인카페': '#90A4AE'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('브랜드별 영업/폐업 현황', fontsize=15, fontweight='bold')

# ── 그래프 1: 프랜차이즈만 (개인카페 제외) ──
franchise_only = df_result_gn[df_result_gn['브랜드'] != '개인카페']
ax = axes[0]
x = range(len(franchise_only))
ax.bar(x, franchise_only['영업중'], label='영업중', color='#2D7A5E')
ax.bar(x, franchise_only['폐업'], label='폐업', color='#E53935',
       bottom=franchise_only['영업중'].values)
ax.set_xticks(x)
ax.set_xticklabels(franchise_only['브랜드'], rotation=15)
ax.set_ylim(0, 90)  
ax.set_title('프랜차이즈만 비교 (개인카페 제외)')
ax.legend()

# ── 그래프 2: 전체 비율(%) 누적 막대 ──
ax = axes[1]
df_pct = df_result_gn.copy()
df_pct['영업비율'] = df_pct['영업중'] / df_pct['전체'] * 100
df_pct['폐업비율'] = df_pct['폐업']   / df_pct['전체'] * 100

x2 = range(len(df_pct))
ax.bar(x2, df_pct['영업비율'], label='영업중', color='#2D7A5E')
ax.bar(x2, df_pct['폐업비율'], label='폐업', color='#E53935',
       bottom=df_pct['영업비율'].values)
ax.set_xticks(x2)
ax.set_xticklabels(df_pct['브랜드'], rotation=15)
ax.set_ylim(0, 110)
ax.set_ylabel('비율 (%)')
ax.set_title('전체 브랜드 비율(%) 비교')
ax.legend()

plt.tight_layout()
plt.savefig('강남구_영업폐업(개인카페 제외).png', dpi=150)
plt.show()

## 2. 성동구 분석

## 1. 라이브러리 & 데이터 로딩

In [ ]:
# ✅ 파일명 수정 (언더바 사용)
data_sd = pd.read_csv('서울시 성동구 휴게음식점 인허가 정보.csv', encoding='EUC-KR')
print("총 데이터 수:", len(data_sd))
data_sd.head()

## 2. 전처리 & 프랜차이즈 분류

In [ ]:
cols = ['사업장명', '상세영업상태명', '인허가일자', '폐업일자', '도로명주소', '좌표정보(X)', '좌표정보(Y)']
data_sd = data_sd[cols].copy()

# 날짜 변환
data_sd['인허가일자'] = pd.to_datetime(data_sd['인허가일자'], errors='coerce')
data_sd['폐업일자']  = pd.to_datetime(data_sd['폐업일자'],  errors='coerce')

# ✅ 프랜차이즈 키워드 수정 (실제 데이터 기준)
franchise_keywords = {
    '스타벅스': '스타벅스',
    '메가커피':  '메가엠지씨',   # ✅ 수정: '메가엠지씨커피' 실제 이름
    '컴포즈':   '컴포즈',
    '빽다방':   '빽다방',
    '투썸':     '투썸플레이스',  # ✅ 수정: '투썸플레이스' 실제 이름
}

def classify(name):
    for brand, keyword in franchise_keywords.items():
        if keyword in str(name):
            return brand
    return '개인카페'

data_sd['브랜드'] = data_sd['사업장명'].apply(classify)

# 브랜드별 수량 확인
print(data_sd['브랜드'].value_counts())

## 3. 브랜드별 추출

In [ ]:
# 스타벅스
starbucks = data_sd[data_sd['브랜드'] == '스타벅스'].copy()
print(f"성동구 스타벅스 총 {len(starbucks)}개")
print(starbucks[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 메가커피
mega = data_sd[data_sd['브랜드'] == '메가커피'].copy()
print(f"성동구 메가커피 총 {len(mega)}개")
print(mega[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 컴포즈
compose = data_sd[data_sd['브랜드'] == '컴포즈'].copy()
print(f"성동구 컴포즈 총 {len(compose)}개")
print(compose[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 빽다방
baek = data_sd[data_sd['브랜드'] == '빽다방'].copy()
print(f"성동구 빽다방 총 {len(baek)}개")
print(baek[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

## 4. 생존율 분석

In [ ]:
brands = ['스타벅스', '메가커피', '컴포즈', '빽다방', '투썸', '개인카페']
result_sd = []

for brand in brands:
    df_b = data_sd[data_sd['브랜드'] == brand]
    total  = len(df_b)
    active = (df_b['상세영업상태명'] == '영업').sum()
    closed = (df_b['상세영업상태명'] == '폐업').sum()
    survival_rate_sd = active / total * 100 if total > 0 else 0

    # 평균 영업 기간 (폐업한 가게 기준)
    closed_df = df_b[df_b['폐업일자'].notna() & df_b['인허가일자'].notna()]
    avg_days = (closed_df['폐업일자'] - closed_df['인허가일자']).dt.days.mean()

    result_sd.append({
        '브랜드': brand,
        '전체': total,
        '영업중': active,
        '폐업': closed,
        '생존율(%)': round(survival_rate_sd, 1),
        '평균영업기간(일)': round(avg_days, 0) if pd.notna(avg_days) else None
    })

df_result_sd = pd.DataFrame(result_sd)
print(df_result_sd.to_string(index=False))

## 5. 시각화 (4종 그래프)

In [ ]:
colors = {
    '스타벅스': '#00704A',
    '메가커피':  '#FF6B35',
    '컴포즈':   '#6A1B9A',
    '빽다방':   '#FFEA00',
    '투썸':     '#FF0000',
    '개인카페':  '#90A4AE'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('성동구 카페 프랜차이즈 vs 개인카페 분석', fontsize=16, fontweight='bold')

# 그래프 1: 브랜드별 생존율
ax = axes[0][0]
bars = ax.bar(df_result_sd['브랜드'], df_result_sd['생존율(%)'],
              color=[colors[b] for b in df_result_sd['브랜드']])
for bar, val in zip(bars, df_result_sd['생존율(%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', fontsize=10)
ax.set_title('브랜드별 생존율 비교')
ax.set_ylabel('생존율 (%)')
ax.set_ylim(0, 115)
ax.tick_params(axis='x', rotation=15)

# 그래프 2: 영업 vs 폐업 누적 막대
ax = axes[0][1]
x = range(len(df_result_sd))
ax.bar(x, df_result_sd['영업중'], label='영업중', color='#2D7A5E')
ax.bar(x, df_result_sd['폐업'], label='폐업', color='#E53935',
       bottom=df_result_sd['영업중'])
ax.set_xticks(x)
ax.set_xticklabels(df_result_sd['브랜드'], rotation=15)
ax.set_title('브랜드별 영업/폐업 현황')
ax.legend()

# 그래프 3: 평균 영업 기간
ax = axes[1][0]
df_days = df_result_sd.dropna(subset=['평균영업기간(일)'])
ax.barh(df_days['브랜드'], df_days['평균영업기간(일)'],
        color=[colors[b] for b in df_days['브랜드']])
ax.set_title('폐업 카페 평균 영업기간 (일)')
ax.set_xlabel('일수')

# 그래프 4: 연도별 신규 개업 추이
ax = axes[1][1]
data_sd['개업연도'] = data_sd['인허가일자'].dt.year
yearly = data_sd.groupby(['개업연도', '브랜드']).size().unstack(fill_value=0)
for brand in brands:
    if brand in yearly.columns:
        ax.plot(yearly.index, yearly[brand], marker='o',
                color=colors[brand], label=brand, linewidth=1.8)
ax.set_title('연도별 신규 개업 추이')
ax.set_xlabel('연도')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('성동구_카페_분석.png', dpi=150)
plt.show()
print("그래프 저장 완료: 성동구_카페_분석.png")

In [ ]:
colors = {
    '스타벅스': '#00704A', '메가커피': '#FF6B35',
    '컴포즈': '#6A1B9A', '빽다방': '#FFEA00',
    '투썸': '#FF0000', '개인카페': '#90A4AE'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('브랜드별 영업/폐업 현황', fontsize=15, fontweight='bold')

# ── 그래프 1: 프랜차이즈만 (개인카페 제외) ──
franchise_only = df_result_sd[df_result_sd['브랜드'] != '개인카페']
ax = axes[0]
x = range(len(franchise_only))
ax.bar(x, franchise_only['영업중'], label='영업중', color='#2D7A5E')
ax.bar(x, franchise_only['폐업'], label='폐업', color='#E53935',
       bottom=franchise_only['영업중'].values)
ax.set_xticks(x)
ax.set_xticklabels(franchise_only['브랜드'], rotation=15)
ax.set_ylim(0, 90)  
ax.set_title('프랜차이즈만 비교 (개인카페 제외)')
ax.legend()

# ── 그래프 2: 전체 비율(%) 누적 막대 ──
ax = axes[1]
df_pct = df_result_sd.copy()
df_pct['영업비율'] = df_pct['영업중'] / df_pct['전체'] * 100
df_pct['폐업비율'] = df_pct['폐업']   / df_pct['전체'] * 100

x2 = range(len(df_pct))
ax.bar(x2, df_pct['영업비율'], label='영업중', color='#2D7A5E')
ax.bar(x2, df_pct['폐업비율'], label='폐업', color='#E53935',
       bottom=df_pct['영업비율'].values)
ax.set_xticks(x2)
ax.set_xticklabels(df_pct['브랜드'], rotation=15)
ax.set_ylim(0, 110)
ax.set_ylabel('비율 (%)')
ax.set_title('전체 브랜드 비율(%) 비교')
ax.legend()

plt.tight_layout()
plt.savefig('성동구_영업폐업(개인카페 제외).png', dpi=150)
plt.show()

## 3. 마포구 분석

In [ ]:
# ✅ 파일명 수정 (언더바 사용)
data_mp = pd.read_csv('서울시 마포구 휴게음식점 인허가 정보.csv', encoding='EUC-KR')
print("총 데이터 수:", len(data_mp))
data_mp.head()

In [ ]:
cols = ['사업장명', '상세영업상태명', '인허가일자', '폐업일자', '도로명주소', '좌표정보(X)', '좌표정보(Y)']
data_mp = data_mp[cols].copy()

# 날짜 변환
data_mp['인허가일자'] = pd.to_datetime(data_mp['인허가일자'], errors='coerce')
data_mp['폐업일자']  = pd.to_datetime(data_mp['폐업일자'],  errors='coerce')

# ✅ 프랜차이즈 키워드 수정 (실제 데이터 기준)
franchise_keywords = {
    '스타벅스': '스타벅스',
    '메가커피':  '메가엠지씨',   # ✅ 수정: '메가엠지씨커피' 실제 이름
    '컴포즈':   '컴포즈',
    '빽다방':   '빽다방',
    '투썸':     '투썸플레이스',  # ✅ 수정: '투썸플레이스' 실제 이름
}

def classify(name):
    for brand, keyword in franchise_keywords.items():
        if keyword in str(name):
            return brand
    return '개인카페'

data_mp['브랜드'] = data_mp['사업장명'].apply(classify)

# 브랜드별 수량 확인
print(data_mp['브랜드'].value_counts())

In [ ]:
# 스타벅스
starbucks = data_mp[data_mp['브랜드'] == '스타벅스'].copy()
print(f"마포구 스타벅스 총 {len(starbucks)}개")
print(starbucks[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 메가커피
mega = data_mp[data_mp['브랜드'] == '메가커피'].copy()
print(f"마포구 메가커피 총 {len(mega)}개")
print(mega[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 컴포즈
compose = data_mp[data_mp['브랜드'] == '컴포즈'].copy()
print(f"마포구 컴포즈 총 {len(compose)}개")
print(compose[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
# 빽다방
baek = data_mp[data_mp['브랜드'] == '빽다방'].copy()
print(f"마포구 빽다방 총 {len(baek)}개")
print(baek[['사업장명', '상세영업상태명', '인허가일자', '폐업일자']].to_string())

In [ ]:
brands = ['스타벅스', '메가커피', '컴포즈', '빽다방', '투썸', '개인카페']
result_mp = []

for brand in brands:
    df_b = data_mp[data_mp['브랜드'] == brand]
    total  = len(df_b)
    active = (df_b['상세영업상태명'] == '영업').sum()
    closed = (df_b['상세영업상태명'] == '폐업').sum()
    survival_rate_mp = active / total * 100 if total > 0 else 0

    # 평균 영업 기간 (폐업한 가게 기준)
    closed_df = df_b[df_b['폐업일자'].notna() & df_b['인허가일자'].notna()]
    avg_days = (closed_df['폐업일자'] - closed_df['인허가일자']).dt.days.mean()

    result_mp.append({
        '브랜드': brand,
        '전체': total,
        '영업중': active,
        '폐업': closed,
        '생존율(%)': round(survival_rate_mp, 1),
        '평균영업기간(일)': round(avg_days, 0) if pd.notna(avg_days) else None
    })

df_result_mp = pd.DataFrame(result_mp)
print(df_result_mp.to_string(index=False))

In [ ]:
colors = {
    '스타벅스': '#00704A',
    '메가커피':  '#FF6B35',
    '컴포즈':   '#6A1B9A',
    '빽다방':   '#FFEA00',
    '투썸':     '#FF0000',
    '개인카페':  '#90A4AE'
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('마포구 카페 프랜차이즈 vs 개인카페 분석', fontsize=16, fontweight='bold')

# 그래프 1: 브랜드별 생존율
ax = axes[0][0]
bars = ax.bar(df_result_mp['브랜드'], df_result_mp['생존율(%)'],
              color=[colors[b] for b in df_result_mp['브랜드']])
for bar, val in zip(bars, df_result_mp['생존율(%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val}%', ha='center', fontsize=10)
ax.set_title('브랜드별 생존율 비교')
ax.set_ylabel('생존율 (%)')
ax.set_ylim(0, 115)
ax.tick_params(axis='x', rotation=15)

# 그래프 2: 영업 vs 폐업 누적 막대
ax = axes[0][1]
x = range(len(df_result_mp))
ax.bar(x, df_result_mp['영업중'], label='영업중', color='#2D7A5E')
ax.bar(x, df_result_mp['폐업'], label='폐업', color='#E53935',
       bottom=df_result_mp['영업중'])
ax.set_xticks(x)
ax.set_xticklabels(df_result_mp['브랜드'], rotation=15)
ax.set_title('브랜드별 영업/폐업 현황')
ax.legend()

# 그래프 3: 평균 영업 기간
ax = axes[1][0]
df_days = df_result_mp.dropna(subset=['평균영업기간(일)'])
ax.barh(df_days['브랜드'], df_days['평균영업기간(일)'],
        color=[colors[b] for b in df_days['브랜드']])
ax.set_title('폐업 카페 평균 영업기간 (일)')
ax.set_xlabel('일수')

# 그래프 4: 연도별 신규 개업 추이
ax = axes[1][1]
data_mp['개업연도'] = data_mp['인허가일자'].dt.year
yearly = data_mp.groupby(['개업연도', '브랜드']).size().unstack(fill_value=0)
for brand in brands:
    if brand in yearly.columns:
        ax.plot(yearly.index, yearly[brand], marker='o',
                color=colors[brand], label=brand, linewidth=1.8)
ax.set_title('연도별 신규 개업 추이')
ax.set_xlabel('연도')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('마포구_카페_분석.png', dpi=150)
plt.show()
print("그래프 저장 완료: 마포구_카페_분석.png")

In [ ]:
colors = {
    '스타벅스': '#00704A', '메가커피': '#FF6B35',
    '컴포즈': '#6A1B9A', '빽다방': '#FFEA00',
    '투썸': '#FF0000', '개인카페': '#90A4AE'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('브랜드별 영업/폐업 현황', fontsize=15, fontweight='bold')

# ── 그래프 1: 프랜차이즈만 (개인카페 제외) ──
franchise_only = df_result_mp[df_result_mp['브랜드'] != '개인카페']
ax = axes[0]
x = range(len(franchise_only))
ax.bar(x, franchise_only['영업중'], label='영업중', color='#2D7A5E')
ax.bar(x, franchise_only['폐업'], label='폐업', color='#E53935',
       bottom=franchise_only['영업중'].values)
ax.set_xticks(x)
ax.set_xticklabels(franchise_only['브랜드'], rotation=15)
ax.set_ylim(0, 90)  
ax.set_title('프랜차이즈만 비교 (개인카페 제외)')
ax.legend()

# ── 그래프 2: 전체 비율(%) 누적 막대 ──
ax = axes[1]
df_pct = df_result_mp.copy()
df_pct['영업비율'] = df_pct['영업중'] / df_pct['전체'] * 100
df_pct['폐업비율'] = df_pct['폐업']   / df_pct['전체'] * 100

x2 = range(len(df_pct))
ax.bar(x2, df_pct['영업비율'], label='영업중', color='#2D7A5E')
ax.bar(x2, df_pct['폐업비율'], label='폐업', color='#E53935',
       bottom=df_pct['영업비율'].values)
ax.set_xticks(x2)
ax.set_xticklabels(df_pct['브랜드'], rotation=15)
ax.set_ylim(0, 110)
ax.set_ylabel('비율 (%)')
ax.set_title('전체 브랜드 비율(%) 비교')
ax.legend()

plt.tight_layout()
plt.savefig('마포구_영업폐업(개인카페 제외).png', dpi=150)
plt.show()

## 4. 종합 비교

In [ ]:
# 세 구 결과 통합
df_result_gn['지역'] = '강남구'
df_result_sd['지역'] = '성동구'
df_result_mp['지역'] = '마포구'
df_combined = pd.concat([df_result_gn, df_result_sd, df_result_mp], ignore_index=True)

# 지역별 브랜드 생존율 비교
brand_data = df_combined[df_combined['브랜드'] != '개인카페']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 브랜드(프랜차이즈) 생존율 비교
brand_pivot = brand_data.groupby(['지역', '브랜드'])['생존율(%)'].mean().reset_index()
brand_names = sorted(brand_pivot['브랜드'].unique())
regions = ['강남구', '성동구', '마포구']
colors = ['#2196F3', '#4CAF50', '#FF9800']
n_brands = len(brand_names)
width = 0.25

for i, (region, color) in enumerate(zip(regions, colors)):
    region_data = brand_pivot[brand_pivot['지역'] == region].set_index('브랜드')
    vals = [region_data.loc[b, '생존율(%)'] if b in region_data.index else 0 for b in brand_names]
    axes[0].bar([j + i * width for j in range(n_brands)], vals,
                width=width, label=region, color=color, alpha=0.8)

tick_pos = [j + width for j in range(n_brands)]
axes[0].set_xticks(tick_pos)
axes[0].set_xticklabels(brand_names, rotation=45, ha='right')
axes[0].set_title('지역별 프랜차이즈 브랜드 생존율 비교', fontsize=14, fontweight='bold')
axes[0].set_xlabel('브랜드')
axes[0].set_ylabel('생존율 (%)')
axes[0].legend()

# 개인카페 생존율 비교
personal_data = df_combined[df_combined['브랜드'] == '개인카페'][['지역', '생존율(%)']].drop_duplicates()
axes[1].bar(personal_data['지역'], personal_data['생존율(%)'],
            color=colors, alpha=0.8, edgecolor='black')
axes[1].set_title('지역별 개인카페 생존율 비교', fontsize=14, fontweight='bold')
axes[1].set_xlabel('지역')
axes[1].set_ylabel('생존율 (%)')
for i, (_, row) in enumerate(personal_data.iterrows()):
    axes[1].text(i, row['생존율(%)'] + 0.5, f"{row['생존율(%)']:.1f}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
print(df_combined.to_string(index=False))


## 결론

- 세 지역(강남구, 성동구, 마포구) 모두 프랜차이즈 카페가 개인카페보다 높은 생존율을 보이며, 브랜드 인지도와 운영 시스템이 생존에 유리하게 작용함을 확인하였다.
- 강남구는 높은 상권 경쟁으로 인해 개인카페 생존율이 세 지역 중 가장 낮은 편이며, 마포구는 문화·주거 혼합 지역 특성상 개인카페의 생존율이 상대적으로 높게 나타났다.
- 카페 창업 시 프랜차이즈 모델을 선택하거나, 상권 특성에 맞는 차별화 전략을 수립하는 것이 장기 생존에 중요한 요소임을 시사한다.